In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

CODE = next(
    path.parent.parent
    for path in Path("/kaggle/input").rglob("harness.py")
    if path.parent.name == "octlm"
)
OUT = Path("/kaggle/working/day8")
OUT.mkdir(parents=True, exist_ok=True)
MODEL = OUT / "qwen"
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print(
    "Code:", CODE, "PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "tokenizers==0.22.2", "jinja2>=3.1.6,<4"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "octlm.day6", "prepare", "--model", str(MODEL)], cwd=CODE, check=True
)

In [ ]:
def run(module, stage, name, *arguments):
    subprocess.run(
        [sys.executable, "-m", module, stage, "--out", str(OUT / f"{name}.jsonl"), *arguments],
        cwd=CODE,
        check=True,
    )


run("octlm.harness", "validate", "validate-eval")
run("octlm.harness", "validate", "validate-train", "--tasks", "fixtures/day8/tasks.json")
run("octlm.day8", "traces", "traces", "--model", str(MODEL))
for seed in (0, 1):
    adapter = str(OUT / f"adapter-{seed}.pt")
    common = ("--model", str(MODEL), "--device", "cuda:0", "--adapter", adapter)
    run("octlm.day8", "train", f"train-{seed}", "--seed", str(seed), *common)
    run("octlm.day8", "eval", f"eval-{seed}", *common)
    print((OUT / f"eval-{seed}.jsonl").read_text().splitlines()[-1], flush=True)
run(
    "octlm.day8",
    "eval",
    "eval-0-int8",
    "--model",
    str(MODEL),
    "--device",
    "cuda:0",
    "--adapter",
    str(OUT / "adapter-0.pt"),
    "--int8",
)
print((OUT / "eval-0-int8.jsonl").read_text().splitlines()[-1], flush=True)